In [0]:
%sql
select * from zara_lakehouse.retail_bronze.products_sold_bronze

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
df = spark.sql("select * from zara_lakehouse.retail_bronze.products_sold_bronze")

In [0]:
duplicates_time_df = df.groupBy("product_id", "city", "time") \
    .count() \
    .filter("count > 1")

In [0]:
display(duplicates_time_df)

In [0]:
df_time = df.withColumn("timestamp", F.to_timestamp("time"))

In [0]:
display(df_time)

In [0]:
duplicates_df = df_time.groupBy("product_id", "city", "pieces_sold") \
    .count() \
    .filter("count > 1")

In [0]:
display(duplicates_df)

In [0]:
df_filtered = df_time \
             .filter(F.col("Product_id") == 185102)  \
             .filter(F.col("city") == "San Antonio") \
             .select("product_id", "city", "pieces_sold", "timestamp")
    
display(df_filtered)

In [0]:
df_date = df_time \
            .withColumn("date", F.date_format(F.to_utc_timestamp("timestamp", "UTC"), "yyyy-MM-dd")) \
            .withColumn("year_month", F.date_format(F.to_utc_timestamp("timestamp", "UTC"), "yyyy-MM"))


In [0]:
display(df_date)

In [0]:
duplicates_df_date = df_date.groupBy("product_id", "city", "pieces_sold", "date") \
    .count() \
    .filter("count > 1")

In [0]:
total = duplicates_df_date.count()
print(total)
display(duplicates_df_date)

In [0]:
duplicates_df_date_withoutpieces = df_date.groupBy("product_id", "city", "date") \
    .count() \
    .filter("count > 1")

In [0]:
total = duplicates_df_date_withoutpieces.count()
print(total)
display(duplicates_df_date_withoutpieces)

In [0]:
window_spec = Window.partitionBy("product_id", "city", "pieces_sold", "date").orderBy("date")

df_window = df_date.withColumn("row_num", F.row_number().over(window_spec)) \
              .withColumn("rank_num", F.rank().over(window_spec)) \
              .withColumn("is_duplicate", F.col("row_num") > 1)

display(df_dd)

In [0]:
df_filtered_duplicate_Removal = df_window \
             .filter(F.col("Product_id") == 188771)  \
             .filter(F.col("city") == "Dallas") \
             .select("product_id", "city", "pieces_sold", "date", "row_num", "rank_num", "is_duplicate" )
    
display(df_filtered_duplicate_Removal)

In [0]:
total_rows = df.count()
print(total_rows)

In [0]:
df_result_count = df_window.groupBy("is_duplicate").count().show()

In [0]:
df_null = df_window.withColumn("is_id_null", F.col("product_id").isNull())

In [0]:
display(df_null)

In [0]:
df_null_count = df_null.groupBy("is_id_null").count().show()

In [0]:
summary_df = df_null.agg(
    F.count("*").alias("total_rows"),
    F.sum(F.when(F.col("is_duplicate") == False, 1).otherwise(0)).alias("non_duplicate_rows"),
    F.sum(F.when(F.col("is_duplicate") == True, 1).otherwise(0)).alias("duplicate_rows"),
    F.sum(F.when(F.col("is_id_null") == True, 1).otherwise(0)).alias("null_rows"),
    F.sum(F.when(F.col("is_id_null") == False, 1).otherwise(0)).alias("non_null_rows")
).withColumn(
    "ingestion_time", F.current_timestamp()
)

display (summary_df)